# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is not subscriptable, access properties directly

# Print short info
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets, fields, and columns (@id).
print('Available Record Sets:')
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    print('No record sets defined in the top-level metadata. Attempting fallback via dataset._schema...')
    # Fallback: check underlying full schema for record sets
    full_json = metadata.to_json()
    record_sets = []
    for key in ["recordSet", "recordSets"]:
        if key in full_json:
            record_sets = full_json[key]
            break
    if not record_sets:
        record_sets = [obj for obj in full_json.get('@graph', []) if obj.get('@type', '').endswith('RecordSet')]
    record_set_ids = [rs['@id'] for rs in record_sets]
for rsid in record_set_ids:
    print(f"- {rsid}")

# For each record set, list its fields and columns (@id)
print('\nRecord Set Field and Column Overview:')
for rsid in record_set_ids:
    rs_json = None
    # The object may be in metadata['recordSet'] or in @graph
    for maybe_rs in metadata.to_json().get('recordSet', []) + metadata.to_json().get('@graph', []):
        if maybe_rs.get('@id') == rsid:
            rs_json = maybe_rs
            break
    if rs_json is None: continue
    print(f"\nRecordSet {rsid}:")
    fields = rs_json.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields/Columns (@id):")
    for fld in fields:
        if isinstance(fld, dict):
            print(f"    - {fld.get('@id')}")
        else:
            print(f"    - {fld}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Re-extract list of record set @id's for use
record_sets = record_set_ids

dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set])} records for {record_set}")

# Show columns of the first record set (if available)
if record_sets:
    print(f"Columns for record set {record_sets[0]}:")
    print(dataframes[record_sets[0]].columns.tolist())
    display(dataframes[record_sets[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, we'll analyze a numeric field from the first record set (if available)
import numpy as np

if record_sets:
    rs_id = record_sets[0]  # Pick the first record set
    df = dataframes[rs_id]
    # Find likely numeric field (e.g., MW, coverage, or count columns)
    numeric_field = None
    for col in df.columns:
        try:
            # Heuristic: look for numeric dtype or can be converted
            if np.issubdtype(df[col].dropna().astype(float).dtype, np.number):
                numeric_field = col
                break
        except Exception:
            continue

    if numeric_field is not None:
        print(f"Analyzing field: {numeric_field}")
        # Ensure the field is numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)  # Use upper quartile as example

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df[[numeric_field]].head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical field (e.g., one containing 'description' or 'accession')
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("Could not find a numeric field for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field is not None:
    # Basic distribution plot
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group_field exists, plot means by group if not too many categories
    if group_field and df[group_field].nunique() < 20:
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_means.values, y=group_means.index, orient='h')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load, explore, and analyze a Croissant-described mass spectrometry dataset using the `mlcroissant` library. We:
- Inspected record sets and fields by `@id`
- Loaded data into Pandas DataFrames
- Performed preliminary filtering and normalization on a numeric field
- Visualized the field's distribution and explored statistical groupings

This workflow can be extended for downstream analyses such as protein expression profiling, pattern discovery, or machine learning pipeline integration.